## Chatbot and RAG Evaluation 

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for buidling LLM applications. 

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn: 
1. How to create test datasets 
2. How to run your RAG application on those datasets 
3. How to measure your application's perfomance using different evaluation metrics 

#### Overview 

A typical RAG evaluation workflow consists of three main steps: 
1. Creating a dataset with questions and their expected answers 
2. Running your RAG application on those questions 
3. Using evaluators to measure how well your paplication performed, looking at factors like: 

    - Answer relevance 
    - Answer accuracy 
    - Retrieval quality 

For this tutorial, we'll create and evaluate a bot that answers questions aboiut a few of Lilian Weng's insightful blog posts 

### Chatbot Evaluation 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [3]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['14539ea3-7532-47aa-9bcb-084a0c42d134',
  '39f75bdb-a5cf-41fd-8fec-b9259ea99b7b',
  '709a10b5-7e8c-48f8-a261-295f6c02d33c',
  '1c8eb7be-fbdd-48cc-b49a-208352ff484f',
  '5a461ad1-3e94-47ee-8bc7-85d27330b4a3'],
 'count': 5,
 'as_of': '2026-08-29T04:30:03.559827383Z'}

#### Define Metrics (LLM as a Judge)

In [4]:
from langsmith import wrappers 
from langchain_groq import ChatGroq

eval_instructions = "You are an expert professor sepcialized in grading students' answers to questions."

def correctness(inputs:dict, outputs: dict, reference_outputs: dict) -> bool: 
    user_content = f"""You are grading the following questions: 
    {inputs['question']}
    Here is the real answer: 
    {reference_outputs['answer']}
    You are grading the following predicted answer: 
    {outputs['response']}
    Respond with CORRECT or INCORRECT: 
    Grade: 
    """

    llm  = ChatGroq(
        model = "qwen/qwen3.6-27b", 
        temperature=0)
    response = llm.invoke( [
                {"role": "system", "content": eval_instructions}, 
                {"role": "user", "content": user_content}
            ])

    return response.content.strip() == 'CORRECT'

In [5]:
## Concisions - checks whether the actual output is less than 2X the lenght of the expected result 

def concision(outputs: dict, reference_outputs: dict) -> bool: 
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

#### Run Evaluations 

In [6]:
default_intstructions = "Repond to the users questions in a short, concise manner (one short sentence)"
def my_app(question: str, model: str = "qwen/qwen3.6-27b", instructions: str = default_intstructions):
    llm = ChatGroq(model=model, temperature=0)
    response = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    return response.content

In [7]:
### Call my_app for every datapoints 
def ls_target(inputs: str) -> dict: 
    return {"response": my_app(inputs["question"])}

In [8]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-mini-chatbot"
)

c:\Users\manas\Documents\development\langgraph-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'openai-4o-mini-chatbot-c2fade4d' at:
https://smith.langchain.com/o/8ee69959-5811-4101-9628-f1aaeaf404f4/datasets/4f310036-352f-47ad-a0e0-184d80dcab39/compare?selectedSessions=ae3b340f-0e04-41d2-86cc-95d9a30ec53e




5it [00:10,  2.02s/it]
